# Linear Regression Deep Dive

# Load packages

In [1]:
# Load packages
library(psych); library(EGAnet)
library(GGally); library(ggplot2)
library(tidyverse); library(car)
library(bestNormalize)

ERROR: Error in library(EGAnet): there is no package called ‘EGAnet’


# Motivating Dataset 1: CERN

Research question: For particles with very little mass, can we predict their energy given momentum alone?

## Load in the Data

In [ ]:
# Load CERN data
cern <- read.csv("data/cern.csv")

## Wrangle the Data

1.    Subset to only include `Run` 165617

2.    Select variables of interest: `E1`, `px1`, `py1`, `pz1`

3.    Name final object as `cern_voi` (variables of interest)

In [ ]:
# Obtain variables of interest
cern_voi <- cern[cern$Run == 165617, c("E1", "px1", "py1", "pz1")]

## Visualize Dataset using `ggpairs`

In [ ]:
# Visualize
ggpairs(cern_voi)

## Simple Linear Regression

In [ ]:
lm(E1 ~ px1, data = cern_voi)

<!--- OR ---->

In [ ]:
lm(cern_voi$E1 ~ cern_voi$px1)

<!--- not super informative, though... --->

In [ ]:
summary(lm(E1 ~ px1, data = cern_voi))

## Your Turn

> Perform simple linear regression using the `py1` as the predictor of `E1` (print out the summary)

In [ ]:
summary(lm(E1 ~ py1, data = cern_voi))

## Plotting

In [ ]:
ggplot(data = cern_voi, aes(x = px1, y = E1)) +
  geom_point() +
  labs(x = "Momentum in the x-direction", y = "Energy") +
  geom_smooth(method = lm) +
  theme_bw()

Doesn't seem to be working well...

Let's try using 3D momentum

## Your Turn: Create 3D Momentum

> Compute 3D momentum by taking the square root of the sum of the squares of these three variables: `px1`, `py1`, `pz1`

1.    Square the variables

2.    Sum the squared values

3.    Take the square root of the sum

This computation *can* be done in a single line of code!

*Hint*: you'll need to sum the rows!

> Store the values as `p1` in your `cern_voi` data frame

In [ ]:
cern_voi$p1 <- sqrt(rowSums(cern_voi[, c("px1", "py1", "pz1")]^2))
head(cern_voi)

> Now, perform simple linear regression

In [ ]:
p1_model <- lm(E1 ~ p1, data = cern_voi)
summary(p1_model)

> Finally, plot

In [ ]:
ggplot(cern_voi, aes(x = p1, y = E1)) +
  geom_point() +
  geom_smooth(method = lm) +
  labs(x = "3D Momentum", y = "Energy") +
  theme_bw()

> Reflect: Why would you expect such different results based on one direction of momentum (`px1` or x-direction) versus three-dimensional momentum (`p1`)?

**Solution:** Energy depends on the magnitude of the particle momentum, not on a single directional component. A particle can have substantial momentum in the y- or z-direction even when `px1` is small, so `px1` alone discards relevant information. The 3D magnitude `p1` combines all three components and should therefore track energy much more closely.




<!--- Back to slides --->

## Assumptions and Diagnostics

Let's plot all three directions of `p`

In [ ]:
# Wrangle into form
cern_voi %>%
  pivot_longer(
    cols = c("px1", "py1", "pz1"),
    names_to = "Direction",
    values_to = "Magnitude"
  ) %>%
ggplot(aes(x = Magnitude, y = E1, group = Direction, color = Direction)) +
  geom_point() +
  labs(x = "Momentum", y = "Energy") +
  geom_smooth(method = lm) +
  theme_bw()

Now, let's make a linear model of momentum to predict energy

In [ ]:
# Let's store our model
E1_model <- lm(E1 ~ px1 + py1 + pz1, data = cern_voi)

# And print the summary
summary(E1_model)

To plot, we need to get our predictions

In [ ]:
# Get model predictions
cern_voi$E1_hat <- predict(E1_model)

# Make a plot
ggplot(data = cern_voi, aes(x = E1_hat, y = E1)) +
  geom_point() +
  labs(x = "Predicted Energy", y = "Observed Energy") +
  geom_smooth(method = lm) +
  theme_bw()

> What do you notice about this plot (e.g., do our predictions seem to fit)?

**Solution:** The observed-versus-predicted plot should be inspected against the 45-degree ideal. The predictions capture some systematic variation in energy, but using the three directional components as separate linear predictors is not equivalent to using the nonlinear momentum magnitude $\sqrt{p_x^2+p_y^2+p_z^2}$, so departures from a perfect fit are expected.




<!--- Back to slides --->

### Multicollinearity

In [ ]:
# Compute VIF (variance inflation factor)
vif(E1_model)

### Outliers

To detect outliers, we'll use three common and related metrics: difference-in-fits (or `dffits`), Cook's distance, and difference-in-betas (or `dfbetas`)

Compute statistics

In [ ]:
# Get influence measures
E1_influence <- influence.measures(E1_model)

# Get data frame
influence_df <- as.data.frame(E1_influence$infmat)

#### difference-in-fits

Compute statistic

In [ ]:
# Plot difference-in-fits with spikes
plot(influence_df$dffit, type = "h")
text(influence_df$dffit, row.names(influence_df))
# Add row names to identify which cases

A general guideline is:

$$
|DFFITS_i| > 2 \cdot \sqrt{\frac{p + 1}{n}},
$$

where $p$ is the number of predictors and $n$ is the sample size.

> Compute the above equation and use `which()` to identify outlying indices

In [ ]:
n <- nrow(cern_voi)
p <- 3
dffits_cutoff <- 2 * sqrt((p + 1) / n)
dffits_cutoff
which(abs(influence_df$dffit) > dffits_cutoff)

> Do there appear to be any outliers?

**Solution:** Use the indices printed by the preceding cutoff calculation. Any listed cases exceed the stated influence cutoff and should be investigated; they should not be deleted automatically.



#### Cook's distance

Compute statistic

In [ ]:
# Plot Cook's distance with spikes
plot(influence_df$cook.d, type = "h")
text(influence_df$cook.d, row.names(influence_df))
# Add row names to identify which cases

A general guideline is:

$$
D > \frac{4}{n - k - 1}
$$

> Compute the above equation and use `which()` to identify outlying indices

In [ ]:
n <- nrow(cern_voi)
k <- 3
cooks_cutoff <- 4 / (n - k - 1)
cooks_cutoff
which(influence_df$cook.d > cooks_cutoff)

> Do there appear to be any outliers?

**Solution:** Use the indices printed by the preceding cutoff calculation. Any listed cases exceed the stated influence cutoff and should be investigated; they should not be deleted automatically.

Fewer outliers than DFFIT but still many outliers.



#### difference-in-betas

Compute statistic

In [ ]:
## Plot difference-in-betas with spikes
plot(influence_df$dfb.1_, type = "h")
text(influence_df$dfb.1_, row.names(cern_voi))
## Add row names to identify which cases

A general guideline is:

$$
|dfbetas| > \frac{2}{\sqrt{n}},
$$

where $p$ is the number of predictors and $n$ is the sample size.

> Compute the above equation and use `which()` to identify outlying indices

In [ ]:
n <- nrow(cern_voi)
dfbetas_cutoff <- 2 / sqrt(n)
dfbetas_cutoff
which(abs(influence_df$dfb.1_) > dfbetas_cutoff)

> Was there a consensus across the influence measures? Do there appear to be any outliers?

**Solution:** Compare the cases flagged by DFFITS, Cook's distance, and DFBETAS. Cases flagged by multiple measures deserve the most attention because the diagnostics agree that they have unusual influence on the fitted model. A flagged case is a prompt for investigation, not automatic removal.



### Normal Residuals

<!---

Shapiro-Wilks test of normality has the following hypotheses

Null (H_0): The values are not significantly different from normal

Alternative (H_A): The values are significantly different from normal

--->

> Before running this code, what do you expect to happen?

A.    Retain the null: The residuals are not significantly different from normal

B.    Reject the null: The residuals are significantly different from normal

In [ ]:
# Perform Shapiro-Wilks test
shapiro.test(residuals(E1_model))

# Plot the residuals
plot(E1_model, which = 1)

### Homoscedasticity

In [ ]:
# Plot the QQ plot
plot(E1_model, which = 2)

## Resolutions

We'll start and work backwards on the resolutions for the linear model assumptions.

### Homoscedasticity

If homoscedasticity is an issue, then we can estimate corrected standard errors using the \{sandwich\} package

In [ ]:
# Load {lmtest} and {sandwich}
library(lmtest); library(sandwich)

# The `coeftest` will print corrected standard errors
# (this correction will affected our *p*-values)
coeftest(E1_model, vcov = vcovHC(E1_model))

> Did anything change?

**Solution:** The coefficient estimates remain the same because the fitted OLS line has not changed. The heteroscedasticity-consistent covariance estimate changes the standard errors and therefore can change the test statistics, p-values, and confidence intervals.



We'll also check out whether there is independence using the Durbin-Watson test in the \{car\} package

In [ ]:
# We'll need to set a seed first
set.seed(42)

# Perform the test
durbinWatsonTest(E1_model)

Here's how to interpret the test:

$< 2$ = positive autocorrelation
$2$ = no autocorrelation
$> 2$ = negative autocorrelation

## Your Turn: Predicting Life Expectancy

Your goal is to work through a similar pipeline as we just walked through above.

Our objective will be to build a model to predict life expectancy.

> Let's start by loading in the dataset: `life_expectancy.csv`

In [ ]:
life <- read.csv("life_expectancy.csv")
dim(life)
names(life)

Here's our variables of interest:

`Life.expectancy` (numeric; 48.1-89.0) = average life expectancy in years

`Alcohol` (numeric; 0.01-15.19) = per capita consumption in liters of pure alcohol

`Schooling` (numeric; 4.9-20.4) = average number of years in school
 
`percentage.expenditure` (numeric; 0-100) = expenditure on health as a percentage of GDP per capita

`GDP` (numeric) = gross domestic product in millions (USD)

> Select these variables and rename them as you see fit

In [ ]:
life_voi <- life %>%
  transmute(
    life_expectancy = Life.expectancy,
    alcohol = Alcohol,
    schooling = Schooling,
    health_expenditure = percentage.expenditure,
    gdp = GDP
  ) %>%
  drop_na()

glimpse(life_voi)

> Run `ggpairs()` to get an idea of the relationships and distributions

In [ ]:
ggpairs(life_voi)

> Reflect on the distributions -- do all variables appear to be normal?

**Solution:** No. In the supplied life-expectancy data, GDP and percentage health expenditure are strongly right-skewed, while the other variables are closer to symmetric but are not all normally distributed. Predictor normality is not itself an OLS requirement; the plots motivate later checks and possible transformations.



We'll come back to these distributions later. For now, let's try to run our model

> Perform linear regression and print the summary

In [ ]:
life_model <- lm(life_expectancy ~ alcohol + schooling + health_expenditure + gdp, data = life_voi)
summary(life_model)

### Check for assumptions (follow workflow above)

> Multicollinearity?

In [ ]:
vif(life_model)

> Any multicollinearity? If so, remove a variable and re-run the model

In [ ]:
# GDP and health_expenditure are strongly collinear; retain GDP and remove health_expenditure.
life_model2 <- lm(life_expectancy ~ alcohol + schooling + gdp, data = life_voi)
summary(life_model2)
vif(life_model2)

> Did the results change?

**Solution:** Yes, but the substantive conclusion is similar: schooling remains the dominant predictor. GDP and percentage health expenditure are strongly redundant (their correlation is about 0.91 and their initial VIFs are about 6), so retaining one removes that source of multicollinearity and changes the conditional coefficient estimates and their standard errors.




> Outliers?

In [ ]:
life_inf <- influence.measures(life_model2)
life_inf_df <- as.data.frame(life_inf$infmat)
n <- nrow(life_voi); k <- 3
dffits_cut <- 2 * sqrt((k + 1) / n)
cooks_cut <- 4 / (n - k - 1)
dfb_cut <- 2 / sqrt(n)
idx_dffits <- which(abs(life_inf_df$dffit) > dffits_cut)
idx_cooks <- which(life_inf_df$cook.d > cooks_cut)
idx_dfbetas <- which(apply(abs(life_inf_df[, grepl("^dfb", names(life_inf_df))]), 1, max) > dfb_cut)
sort(unique(c(idx_dffits, idx_cooks, idx_dfbetas)))

> Any outliers? If so, remove observation(s) and re-run the model

In [ ]:
outlier_idx <- sort(unique(c(idx_dffits, idx_cooks, idx_dfbetas)))
life_clean <- life_voi[-outlier_idx, ]
life_model3 <- lm(life_expectancy ~ alcohol + schooling + gdp, data = life_clean)
summary(life_model3)

> Normal residuals?

In [ ]:
shapiro.test(residuals(life_model3))
plot(life_model3, which = 1)

> Are the residuals normal?

**Solution:** For the initial complete-case model, Shapiro--Wilk gives approximately $p=0.005$, providing evidence against normally distributed residuals. After model changes/outlier handling, use the printed test together with the residual and Q--Q plots; a non-significant test alone does not prove normality.



> Homoscedasticity?

In [ ]:
plot(life_model3, which = 2)

Before jumping to outliers, let's revisit the distributions (take another look at the `ggpairs` plot above)

> Let's transform our variables to be more normal

<!--- Hint: one of these variables should have been removed due to multicollinearity --->

We'll use the \{bestNormalize\} package to search over many different transformations to identify the one that gets our data closest to normal. Here's a list of transformations it applies:

arcsinh:

$$
\sinh^{-1}(x)
$$

box-cox:

$$
\begin{cases}
\frac{(y + \lambda_2)^{\lambda_1 - 1}}{\lambda_1}, \text{if} \ \lambda_1 \neq 0 \\
\log(y + \lambda_2), \text{if} \lambda_1 = 0
\end{cases}
$$

center and scale: `scale()`

$$
\frac{x - \bar{x}}{s_x}
$$

double reversed log:

$$
-\log_b(x + a)
$$

exponent:

$$
e^x
$$

log:

$$
\log_b(x + a)
$$

orderNorm:

$$
\Phi^{-1} \left( \frac{\text{rank}(x)}{n + 1} \right)
$$

where $\Phi^{-1}$ is the inverse cumulative normal distribution function (`qnorm()`)

square root:

$$
\sqrt{x + a}
$$

yeo-johnson: too complex to precisely display as in-line text

> Create an object called `life_voi` with the data frame you've been using

In [ ]:
life_voi <- life_clean %>% select(life_expectancy, alcohol, schooling, gdp)

This code is provided for you:

In [ ]:
# Set seed
set.seed(42)

# Get {bestNormalize} objects
life_bestNormalize <- lapply(life_voi, bestNormalize)

# Extract transformed values
life_transformed <- do.call(
  cbind.data.frame, lapply(life_bestNormalize, function(x){x$x.t})
)

Here's how to access which transformations were used (if any):

In [ ]:
# Extract transformation types
lapply(life_bestNormalize, function(x){x$chosen_transform})

> Re-fit your linear model using the `life_transformed` object

In [ ]:
life_transformed_model <- lm(life_expectancy ~ alcohol + schooling + gdp, data = life_transformed)
summary(life_transformed_model)

> Did your model prediction ($R^2$) get worse or better?

**Solution:** Compare the printed $R^2$ with the previous model on the corresponding analysis sample. Transformation can improve distributional behavior without necessarily increasing $R^2$ on the original response scale, so the direction should be reported from the fitted output rather than assumed.



> Interpret the `schooling` parameter. (hint: the values have been standardized!)

**Solution:** Because the transformed/standardized variables are on standardized scales, the schooling coefficient represents the expected change in transformed life expectancy, in standard-deviation units, associated with a one-standard-deviation increase in transformed schooling, holding the other predictors constant.



> Normal residuals?

In [ ]:
# Shapiro-Wilk
shapiro.test(residuals(life_transformed_model))

# Residual plot
plot(life_transformed_model, which = 1)

> Are the residuals normal?

**Solution:** For the initial complete-case model, Shapiro--Wilk gives approximately $p=0.005$, providing evidence against normally distributed residuals. After model changes/outlier handling, use the printed test together with the residual and Q--Q plots; a non-significant test alone does not prove normality.



> Homoscedasticity?

In [ ]:
# QQ-plot
plot(life_transformed_model, which = 2)